<a href="https://colab.research.google.com/github/C026-01-0774-2023-NewtonOlando/Streamlit-Bank-Approval-/blob/main/projectA(LOAN_APPROAVAL).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os

MODEL_DIR = "/content/drive/MyDrive/Loan_Approval_Project/models"

print("Model folder exists:", os.path.exists(MODEL_DIR))

print("\nFiles found:")

if os.path.exists(MODEL_DIR):
    for file in os.listdir(MODEL_DIR):
        print("✅", file)
else:
    print("❌ Model folder not found")

Model folder exists: True

Files found:
✅ loan_approval_logistic_pipeline.pkl
✅ loan_approval_svm_pipeline.pkl
✅ loan_approval_preprocessor_6.pkl


In [3]:
!pip -q install streamlit joblib scikit-learn pandas numpy
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print("✅ Installation complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 113.4 MB/s eta 0:00:00
✅ Installation complete


In [4]:

%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os

# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Loan Approval Prediction",
    page_icon="💳",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700;800&display=swap');

html, body, [class*="css"] {
    font-family: 'Poppins', sans-serif;
}

.stApp {
    background:
        radial-gradient(circle at top left, rgba(30,64,175,.30), transparent 35%),
        radial-gradient(circle at bottom right, rgba(14,116,144,.20), transparent 35%),
        #06111f;
    color: #f1f5f9;
}

.block-container {
    padding-top: 2rem;
    padding-bottom: 3rem;
    max-width: 1400px;
}

.hero {
    padding: 30px;
    border-radius: 24px;
    background: rgba(15,23,42,.72);
    border: 1px solid rgba(96,165,250,.20);
    box-shadow: 0 20px 50px rgba(0,0,0,.35);
    margin-bottom: 25px;
}

.hero h1 {
    margin: 0;
    font-size: 38px;
    font-weight: 800;
    color: #e0f2fe;
}

.hero p {
    margin-top: 8px;
    color: #94a3b8;
    font-size: 15px;
}

.glass-card {
    background: rgba(15,23,42,.68);
    border: 1px solid rgba(148,163,184,.15);
    border-radius: 20px;
    padding: 24px;
    margin-bottom: 20px;
    box-shadow: 0 15px 40px rgba(0,0,0,.25);
}

.section-title {
    font-size: 20px;
    font-weight: 700;
    color: #bae6fd;
    margin-bottom: 15px;
}

.metric-card {
    background: rgba(15,23,42,.75);
    border: 1px solid rgba(96,165,250,.18);
    border-radius: 16px;
    padding: 18px;
    text-align: center;
}

.metric-label {
    color: #94a3b8;
    font-size: 12px;
}

.metric-value {
    color: #e0f2fe;
    font-size: 23px;
    font-weight: 700;
    margin-top: 5px;
}

.approved-card {
    background: rgba(6,78,59,.40);
    border: 1px solid rgba(52,211,153,.40);
    border-radius: 22px;
    padding: 30px;
    text-align: center;
}

.rejected-card {
    background: rgba(127,29,29,.35);
    border: 1px solid rgba(248,113,113,.40);
    border-radius: 22px;
    padding: 30px;
    text-align: center;
}

.prediction-title {
    font-size: 30px;
    font-weight: 800;
}

.probability {
    font-size: 42px;
    font-weight: 800;
    margin: 10px 0;
}

.stButton > button {
    width: 100%;
    border-radius: 12px;
    border: 1px solid rgba(96,165,250,.30);
    background: linear-gradient(135deg,#1d4ed8,#0369a1);
    color: white;
    font-weight: 700;
    padding: 13px;
}

section[data-testid="stSidebar"] {
    background: #071426;
    border-right: 1px solid rgba(148,163,184,.12);
}

.footer {
    text-align: center;
    color: #64748b;
    font-size: 12px;
    margin-top: 35px;
}
</style>
""", unsafe_allow_html=True)


# ============================================================
# MODEL FILES
# ============================================================

MODEL_DIR = "/content/drive/MyDrive/Loan_Approval_Project/models"

LR_PATH = os.path.join(
    MODEL_DIR, "loan_approval_logistic_pipeline.pkl"
)
SVM_PATH = os.path.join(
    MODEL_DIR, "loan_approval_svm_pipeline.pkl"
)
PREPROCESSOR_PATH = os.path.join(
    MODEL_DIR, "loan_approval_preprocessor_6.pkl"
)

THRESHOLD = 0.65


@st.cache_resource
def load_models():
    lr_model = joblib.load(LR_PATH)
    svm_model = joblib.load(SVM_PATH)
    preprocessor = joblib.load(PREPROCESSOR_PATH)
    return lr_model, svm_model, preprocessor


try:
    lr_model, svm_model, preprocessor = load_models()
except Exception as e:
    st.error("Unable to load the loan approval models.")
    st.code(str(e))
    st.stop()


# ============================================================
# ESTIMATED CREDIT SCORE FROM SEVEN QUESTIONS
# ============================================================

def calculate_estimated_credit_score(
    payment_habit,
    late_payment_band,
    previous_default,
    debt_collections,
    credit_history,
    active_loans,
    credit_utilization
):
    """
    Illustrative questionnaire score for a prototype.
    This is NOT a credit-bureau score or a validated
    lending-risk score.
    """

    # Points for each answer. Higher points represent
    # generally more favorable reported behavior.

    payment_points = {
        "Always": 160,
        "Usually": 120,
        "Sometimes": 70,
        "Rarely": 20,
        "Never": 0
    }

    late_points = {
        "Never": 100,
        "1–2 times": 70,
        "3–5 times": 40,
        "6–10 times": 0,
        "More than 10 times": -70
    }

    default_points = {
        "No": 80,
        "Yes": -100
    }

    collections_points = {
        "No": 50,
        "Yes": -60
    }

    history_points = {
        "Less than 1 year": 10,
        "1–3 years": 30,
        "3–5 years": 50,
        "5–10 years": 70,
        "More than 10 years": 90
    }

    loans_points = {
        "None": 50,
        "1": 40,
        "2": 20,
        "3": 0,
        "4 or more": -40
    }

    utilization_points = {
        "Less than 10%": 70,
        "10–30%": 60,
        "31–50%": 40,
        "51–75%": 5,
        "More than 75%": -50
    }

    # Start at 300 and add the questionnaire points.
    raw_score = (
        300
        + payment_points[payment_habit]
        + late_points[late_payment_band]
        + default_points[previous_default]
        + collections_points[debt_collections]
        + history_points[credit_history]
        + loans_points[active_loans]
        + utilization_points[credit_utilization]
    )

    # Map the raw point total to the model's 300–850 range.
    # The raw total can exceed that range, so clamp it.
    estimated_score = int(np.clip(raw_score, 300, 850))

    return estimated_score


# ============================================================
# HEADER
# ============================================================

st.markdown("""
<div class="hero">
    <h1>💳 Loan Approval Intelligence</h1>
    <p>
        Loan approval prediction using Logistic Regression
        and Support Vector Machine models.
    </p>
</div>
""", unsafe_allow_html=True)


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:
    st.markdown("## ⚙️ Model Settings")

    model_choice = st.selectbox(
        "Select Prediction Model",
        ["Logistic Regression", "Support Vector Machine"]
    )

    st.markdown("---")
    st.markdown("### 🎯 Decision Threshold")
    st.info(
        f"Fixed threshold: {THRESHOLD:.2f}\n\n"
        "The model predicts approval when its estimated "
        "approval probability is at least 65%."
    )

    st.markdown("---")
    st.markdown("### 🧠 Features Used by Model")
    st.write("• Debt-to-income ratio")
    st.write("• Estimated credit score")
    st.write("• Late payments in last 2 years")
    st.write("• Bankruptcy history")
    st.write("• Employment type")
    st.write("• Collateral provided")

    st.markdown("---")
    st.caption("Loan Approval Prediction System")


# ============================================================
# PERSONAL INFORMATION
# ============================================================

st.markdown('<div class="glass-card">', unsafe_allow_html=True)
st.markdown(
    '<div class="section-title">👤 Personal Information</div>',
    unsafe_allow_html=True
)

col1, col2, col3 = st.columns(3)

with col1:
    age = st.number_input(
        "Age", min_value=18, max_value=100, value=30
    )

with col2:
    gender = st.selectbox("Gender", ["Female", "Male"])

with col3:
    marital_status = st.selectbox(
        "Marital Status",
        ["Single", "Married", "Divorced", "Widowed"]
    )

st.markdown('</div>', unsafe_allow_html=True)


# ============================================================
# EMPLOYMENT AND INCOME
# ============================================================

st.markdown('<div class="glass-card">', unsafe_allow_html=True)
st.markdown(
    '<div class="section-title">💼 Employment & Income</div>',
    unsafe_allow_html=True
)

col1, col2, col3 = st.columns(3)

with col1:
    employment_type = st.selectbox(
        "Employment Type",
        [
            "Business Owner",
            "Retired",
            "Salaried",
            "Self-Employed",
            "Unemployed"
        ]
    )

with col2:
    monthly_income = st.number_input(
        "Monthly Income (KSh)",
        min_value=0.0,
        value=50000.0,
        step=1000.0
    )

with col3:
    co_applicant_income = st.number_input(
        "Co-Applicant Monthly Income (KSh)",
        min_value=0.0,
        value=0.0,
        step=1000.0
    )

st.markdown('</div>', unsafe_allow_html=True)


# ============================================================
# SEVEN CREDIT ASSESSMENT QUESTIONS
# ============================================================

st.markdown('<div class="glass-card">', unsafe_allow_html=True)
st.markdown(
    '<div class="section-title">📊 Credit Assessment</div>',
    unsafe_allow_html=True
)

st.write(
    "Please answer the following questions about your "
    "payment and borrowing history."
)

col1, col2 = st.columns(2)

with col1:
    payment_habit = st.selectbox(
        "1. How often do you make loan or bill payments on time?",
        ["Always", "Usually", "Sometimes", "Rarely", "Never"]
    )

    late_payment_band = st.selectbox(
        "2. In the past 2 years, how many times have you paid late?",
        ["Never", "1–2 times", "3–5 times",
         "6–10 times", "More than 10"]
    )

    previous_default = st.selectbox(
        "3. Have you ever defaulted on a loan?",
        ["No", "Yes"]
    )

    debt_collections = st.selectbox(
        "4. Has a debt ever been sent to a collections agency?",
        ["No", "Yes"]
    )

with col2:
    credit_history = st.selectbox(
        "5. How long have you had your oldest active bank or credit account?",
        [
            "Less than 1 year",
            "1–3 years",
            "3–5 years",
            "5–10 years",
            "More than 10 years"
        ]
    )

    active_loans = st.selectbox(
        "6. How many loans or credit accounts are you currently paying?",
        ["None", "1", "2", "3", "4 or more"]
    )

    credit_utilization = st.selectbox(
        "7. Approximately how much of your available credit limit are you using?",
        [
            "Less than 10%",
            "10–30%",
            "31–50%",
            "51–75%",
            "More than 75%"
        ]
    )

    bankruptcy_history = st.selectbox(
        "Bankruptcy history",
        ["No", "Yes"]
    )

st.caption(
    "The answers are used to calculate an estimated score for "
    "this prototype. They do not retrieve or verify an official "
    "credit-bureau score."
)

st.markdown('</div>', unsafe_allow_html=True)


# ============================================================
# BANKING INFORMATION
# ============================================================

st.markdown('<div class="glass-card">', unsafe_allow_html=True)
st.markdown(
    '<div class="section-title">🏦 Banking Information</div>',
    unsafe_allow_html=True
)

col1, col2 = st.columns(2)

with col1:
    savings_balance = st.number_input(
        "Savings Balance (KSh)",
        min_value=0.0,
        value=50000.0,
        step=1000.0
    )

with col2:
    existing_emi_monthly = st.number_input(
        "Existing Monthly Loan Payments (KSh)",
        min_value=0.0,
        value=0.0,
        step=500.0
    )

st.markdown('</div>', unsafe_allow_html=True)


# ============================================================
# LOAN INFORMATION
# ============================================================

st.markdown('<div class="glass-card">', unsafe_allow_html=True)
st.markdown(
    '<div class="section-title">💰 Loan Information</div>',
    unsafe_allow_html=True
)

col1, col2, col3 = st.columns(3)

with col1:
    loan_amount_requested = st.number_input(
        "Loan Amount Requested (KSh)",
        min_value=1000.0,
        value=100000.0,
        step=5000.0
    )

with col2:
    loan_term_months = st.number_input(
        "Loan Term (Months)",
        min_value=1,
        max_value=360,
        value=60
    )

with col3:
    interest_rate = st.number_input(
        "Annual Interest Rate (%)",
        min_value=0.0,
        max_value=50.0,
        value=12.0,
        step=0.1
    )

collateral_provided = st.selectbox(
    "Is collateral provided?",
    ["No", "Yes"]
)

st.markdown('</div>', unsafe_allow_html=True)


# ============================================================
# CALCULATE ESTIMATED SCORE
# ============================================================

estimated_credit_score = calculate_estimated_credit_score(
    payment_habit=payment_habit,
    late_payment_band=late_payment_band,
    previous_default=previous_default,
    debt_collections=debt_collections,
    credit_history=credit_history,
    active_loans=active_loans,
    credit_utilization=credit_utilization
)

# Derive the late-payment count expected by the model.
# Use representative counts for grouped answers.
late_payment_count_map = {
    "Never": 0,
    "1–2 times": 1,
    "3–5 times": 4,
    "6–10 times": 8,
    "More than 10": 11
}

late_payments_last_2yrs = late_payment_count_map[
    late_payment_band
]


# ============================================================
# DTI CALCULATION
# ============================================================

def calculate_monthly_payment(principal, annual_rate, months):
    if principal <= 0 or months <= 0:
        return 0.0

    monthly_rate = annual_rate / 100 / 12

    if monthly_rate == 0:
        return principal / months

    return (
        principal * monthly_rate * (1 + monthly_rate) ** months
        / ((1 + monthly_rate) ** months - 1)
    )


proposed_monthly_payment = calculate_monthly_payment(
    loan_amount_requested,
    interest_rate,
    loan_term_months
)

total_monthly_income = monthly_income + co_applicant_income

total_monthly_debt = (
    existing_emi_monthly + proposed_monthly_payment
)

debt_to_income_ratio = (
    total_monthly_debt / total_monthly_income
    if total_monthly_income > 0
    else 0.0
)


# ============================================================
# FINANCIAL ASSESSMENT
# ============================================================

st.markdown('<div class="glass-card">', unsafe_allow_html=True)
st.markdown(
    '<div class="section-title">📈 Financial Assessment</div>',
    unsafe_allow_html=True
)

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.markdown(
        f"""<div class="metric-card">
        <div class="metric-label">Total Monthly Income</div>
        <div class="metric-value">KSh {total_monthly_income:,.0f}</div>
        </div>""",
        unsafe_allow_html=True
    )

with col2:
    st.markdown(
        f"""<div class="metric-card">
        <div class="metric-label">Proposed EMI</div>
        <div class="metric-value">KSh {proposed_monthly_payment:,.0f}</div>
        </div>""",
        unsafe_allow_html=True
    )

with col3:
    st.markdown(
        f"""<div class="metric-card">
        <div class="metric-label">Total Monthly Debt</div>
        <div class="metric-value">KSh {total_monthly_debt:,.0f}</div>
        </div>""",
        unsafe_allow_html=True
    )

with col4:
    st.markdown(
        f"""<div class="metric-card">
        <div class="metric-label">Debt-to-Income Ratio</div>
        <div class="metric-value">{debt_to_income_ratio:.2f}</div>
        </div>""",
        unsafe_allow_html=True
    )

st.markdown('</div>', unsafe_allow_html=True)


# ============================================================
# SIX RAW FEATURES SENT TO ORIGINAL PREPROCESSOR
# ============================================================

model_input = pd.DataFrame({
    "debt_to_income_ratio": [debt_to_income_ratio],
    "credit_score": [estimated_credit_score],
    "late_payments_last_2yrs": [late_payments_last_2yrs],
    "bankruptcy_history": [bankruptcy_history],
    "employment_type": [employment_type],
    "collateral_provided": [collateral_provided]
})

with st.expander("🔍 View the six features supplied to the model"):
    st.dataframe(model_input, use_container_width=True)


# ============================================================
# PREDICTION
# ============================================================

st.markdown('<div class="glass-card">', unsafe_allow_html=True)
st.markdown(
    '<div class="section-title">🤖 Loan Prediction</div>',
    unsafe_allow_html=True
)

predict_button = st.button(
    "🚀 Predict Loan Approval",
    use_container_width=True
)

st.markdown('</div>', unsafe_allow_html=True)


if predict_button:
    try:
        # Apply the original fitted preprocessor.
        processed_input = preprocessor.transform(model_input)

        if model_choice == "Logistic Regression":
            model = lr_model
        else:
            model = svm_model

        probability = float(model.predict_proba(processed_input)[0, 1])

        prediction = (
            "APPROVED" if probability >= THRESHOLD else "REJECTED"
        )

        card_class = (
            "approved-card" if prediction == "APPROVED"
            else "rejected-card"
        )

        icon = "✅" if prediction == "APPROVED" else "❌"

        st.markdown(
            f"""
            <div class="{card_class}">
                <div class="prediction-title">{icon} LOAN {prediction}</div>
                <div class="probability">{probability * 100:.2f}%</div>
                <p>Model-estimated probability of loan approval</p>
                <hr>
                <p>
                    <b>Model:</b> {model_choice}<br>
                    <b>Decision threshold:</b> {THRESHOLD:.2f}<br>
                    <b>DTI:</b> {debt_to_income_ratio:.2f}
                </p>
            </div>
            """,
            unsafe_allow_html=True
        )

        st.markdown("### 🔬 Prediction Details")

        col1, col2, col3 = st.columns(3)

        with col1:
            st.metric("Estimated Credit Score", f"{estimated_credit_score}/850")

        with col2:
            st.metric("Approval Probability", f"{probability * 100:.2f}%")

        with col3:
            st.metric("Decision Threshold", f"{THRESHOLD * 100:.0f}%")

        with st.expander("🧮 View the 12 processed model features"):
            processed_df = pd.DataFrame(
                np.asarray(processed_input),
                columns=preprocessor.get_feature_names_out()
            )
            st.dataframe(processed_df, use_container_width=True)

        st.warning(
            "Prototype output only. The questionnaire-derived score "
            "is not an official credit score, and this prediction "
            "should not be used as the sole basis for a real lending decision."
        )

    except Exception as e:
        st.error("Prediction failed.")
        st.code(str(e))


# ============================================================
# FOOTER
# ============================================================

st.markdown("""
<div class="footer">
Loan Approval Prediction System • Six-Feature Model • Threshold 0.65
</div>
""", unsafe_allow_html=True)


Writing app.py


In [5]:
!pip -q install streamlit joblib scikit-learn pandas numpy

In [6]:
!pkill -f streamlit || true

!streamlit run app.py \
    --server.address 0.0.0.0 \
    --server.port 8501 \
    --server.headless true \
    > streamlit.log 2>&1 &

^C


In [ ]:
!pkill -f cloudflared || true

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared

!chmod +x /usr/local/bin/cloudflared

!cloudflared tunnel --url http://localhost:8501

^C
2026-10-14T08:40:04Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-14T08:40:04Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-14T08:40:08Z INF +--------------------------------------------------------------------------------------------+
2026-10-14T08:40:08Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-14T08:40:08Z INF |  https://answered-firm-acid-cruise.trycloudflare.co